In [4]:
# importing the required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import kmapper as km
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

In [2]:
# importing preprocessed data
combined_df = pd.read_csv("../data/combined_cleaned_encoded.csv")

In [3]:
# Scaling 
# Select numerical columns for scaling
num_cols = ['age', 'tenure', 'monthlycharges']

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on numeric columns and transform them
df_scaled = combined_df.copy()
df_scaled[num_cols] = scaler.fit_transform(combined_df[num_cols])

print(df_scaled.head())

      customer_id  gender       age    tenure  monthlycharges  paymentmethod  \
0  TEL_7590-VHVEG       0 -0.371853 -1.511772       -1.155154              5   
1  TEL_5575-GNVDE       1 -0.103853  0.076123       -0.136535              6   
2  TEL_3668-QPYBK       1 -1.376852 -1.463654       -0.253056              6   
3  TEL_7795-CFOCW       1 -1.443852  0.605421       -0.687190              0   
4  TEL_9237-HQITU       0 -1.577852 -1.463654        0.380292              5   

   churn  industry  
0    0.0         2  
1    0.0         2  
2    1.0         2  
3    0.0         2  
4    1.0         2  


In [19]:
#dropping the customer_id column since it is not a numerical column
df_scaled = df_scaled.drop(columns=["customer_id"])

In [20]:
#preparing scaled data (converting from pandas to numpy to get numerical matrix form)
df_data = df_scaled.values

In [21]:
#creating mapper object (verbose=1 shows progess messages)
mapper = km.KeplerMapper(verbose=1)

KeplerMapper(verbose=1)


In [22]:
#choosing the projection
#we take the scaled dataset, reduce it to 2D using TSNE and it gives projected points so Mapper can 
#build a topological graph
projected = mapper.fit_transform(df_data, projection=TSNE(n_components=2))

..Composing projection pipeline of length 1:
	Projections: TSNE()
	Distance matrices: False
	Scalers: MinMaxScaler()
..Projecting on data shaped (21129, 7)

..Projecting data using: 
	TSNE(verbose=1)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 21129 samples in 0.052s...
[t-SNE] Computed neighbors for 21129 samples in 1.970s...
[t-SNE] Computed conditional probabilities for sample 1000 / 21129
[t-SNE] Computed conditional probabilities for sample 2000 / 21129
[t-SNE] Computed conditional probabilities for sample 3000 / 21129
[t-SNE] Computed conditional probabilities for sample 4000 / 21129
[t-SNE] Computed conditional probabilities for sample 5000 / 21129
[t-SNE] Computed conditional probabilities for sample 6000 / 21129
[t-SNE] Computed conditional probabilities for sample 7000 / 21129
[t-SNE] Computed conditional probabilities for sample 8000 / 21129
[t-SNE] Computed conditional probabilities for sample 9000 / 21129
[t-SNE] Computed conditional probabilities for sample

In [24]:
#creating the topological network
graph=mapper.map(projected, df_data, clusterer=KMeans(n_clusters=4), cover=km.Cover(n_cubes=10, perc_overlap=0.3))

Mapping on data shaped (21129, 7) using lens shaped (21129, 2)

Creating 100 hypercubes.

Created 1176 edges and 340 nodes in 0:00:01.314874.


In [28]:
#extracting TDA features

#graph['nodes'] includes list of sample indices in each node
nodes = graph['nodes']

#creating a feature matrix where each sample gets a binary vector for node members
tda_features = np.zeros((df_data.shape[0], len(nodes)))

for i, node_samples in enumerate(nodes.values()):
    tda_features[node_samples, i] = 1 #marking 1 if the sample belongs to this node


#turning the above into one-hot encoding
tda_features_df = pd.DataFrame(tda_features, columns=[f"tda_node_{i}" for i in range(len(nodes))])

#combining with original data
df_final = pd.concat([pd.DataFrame(df_data).reset_index(drop=True), tda_features_df.reset_index(drop=True)], axis=1)

In [29]:
#downloading the final dataset into data folder

save_path = "../data"

os.makedirs(save_path, exist_ok=True)

df_final.to_csv(os.path.join(save_path, "combined_cleaned_TDA.csv"), index=False)